# Chapter 4 &mdash; Acceptance by Paths

**Concept 3 of the Chapter 4 decomposition:** *Acceptance by Paths; Languages $\emptyset$ and $\{\varepsilon\}$ Read Off the Picture*

A string is accepted if it labels a path from the initial state to a final state. Two degenerate cases fall out at once.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-Acceptance-By-Paths/Concept-Acceptance-By-Paths.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


$w$ is **accepted** if it labels a path from the initial state to a final state &mdash; the
$i$-th edge carrying the $i$-th symbol. The set of accepted strings is the language
**recognised** by the DFA.

Two consequences drop out immediately:

* **no final state** $\Rightarrow$ the language is $\emptyset$;
* **initial state is also final** $\Rightarrow$ there is a path of length 0, so
  $\varepsilon$ is in the language.

## 2. Definitions

### A machine with no final state

In [ ]:
nofinal = md2mc('''DFA
I : 0 | 1 -> I
''')
print("final states :", nofinal["F"])

### A machine whose initial state is final

In [ ]:
epsok = md2mc('''DFA
IF : 0 -> A
A  : 0 -> IF
''')
print("initial == final? ", epsok["q0"] in epsok["F"])

<!-- nav-strip -->

---

&larr;&nbsp;[Ch4&nbsp;2.&nbsp;Elements of a DFA: States, Transitions, Initial and Final States](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-Elements-Of-A-DFA/Concept-Elements-Of-A-DFA.ipynb) &nbsp;&middot;&nbsp; [**Chapter 4** index](https://github.com/ganeshutah/Jove/blob/master/Chapter4-DFA/README.md) &nbsp;&middot;&nbsp; [Ch4&nbsp;4.&nbsp;The Formal Five-Tuple $(Q,\Sigma,\delta,q_0,F)$](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-The-Five-Tuple/Concept-The-Five-Tuple.ipynb)&nbsp;&rarr;

---

## 3. Tests

No final state means the empty language.

In [ ]:
from itertools import product
tested = [''.join(p) for n in range(4) for p in product('01', repeat=n)]
print("accepts anything at all?", any(accepts_dfa(nofinal, s) for s in tested))
assert not any(accepts_dfa(nofinal, s) for s in tested)
print("  -> language is the empty set")

Initial-and-final means $\varepsilon$ is accepted &mdash; a path of length 0.

In [ ]:
print("accepts '' ?", accepts_dfa(epsok, ''))
assert accepts_dfa(epsok, '')
print("  -> the zero-length path from IF to itself")

Tracing a path by hand agrees with `accepts_dfa`.

In [ ]:
def trace(D, s):
    q, path = D["q0"], [D["q0"]]
    for ch in s:
        q = step_dfa(D, q, ch); path.append(q)
    return path, q in D["F"]

p, ok = trace(epsok, '0000')
print("path :", ' -> '.join(p), "   accepted:", ok)
assert ok == accepts_dfa(epsok, '0000')

## 4. Exercises


1. Build a DFA with **all** states final. What language does it recognise?
2. How many distinct paths of length 4 exist in `epsok`? How many end in a final state?
3. Why does "path of length 0" make $\varepsilon$ acceptance automatic rather than a
   special case?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 255 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter4-DFA/Concept-Acceptance-By-Paths')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')